# Pythia-1.4B — Causal Patching

**Phase 3:** Activation patching for Pythia-1.4B.

**Hardware:** A100 80GB  
**Runtime:** ~30 minutes

**Run cells 1-8 in order.**

**Outputs:** `causal_patching_pythia_results.csv`

**Same-group baseline:** layer 12 same-group KL = 0.0002


In [ ]:
# Cell 1 — install if not already done
!pip install -q numpy==1.26.4
!pip install -q transformer_lens
print('Done')

In [ ]:
# Configuration
DRIVE_BASE = '/content/drive/MyDrive/identity-under-the-hood'  # change if needed

import os, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from datasets import load_dataset
from transformer_lens import HookedTransformer
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError:
    pass

OUTPUT_DIR = f'{DRIVE_BASE}/causal_patching'
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TARGET_LAYERS = [6, 12, 18, 23]
N_PAIRS = 10
print(f'Device: {DEVICE}')
print(f'Drive: {DRIVE_BASE}')
print('Ready.')


In [ ]:
# Cell 3 — load BBQ race data
dataset   = load_dataset('Elfsong/BBQ', split='race_ethnicity')
race_df   = dataset.to_pandas()
race_ambig = race_df[race_df['context_condition'] == 'ambig'].reset_index(drop=True)

group_indices = {'Hispanic': [], 'Black': []}
for i, row in race_ambig.iterrows():
    context = row['context']
    if 'Hispanic' in context:
        group_indices['Hispanic'].append(i)
    elif 'Black' in context and 'African American' not in context:
        group_indices['Black'].append(i)

hispanic_rows = [race_ambig.iloc[i] for i in group_indices['Hispanic']]
black_rows    = [race_ambig.iloc[i] for i in group_indices['Black']]
print(f'Hispanic: {len(hispanic_rows)}, Black: {len(black_rows)}')

In [ ]:
# Cell 4 — helpers
def format_prompt(row):
    return (
        f"Context: {row['context']}\n"
        f"Question: {row['question']}\n"
        f"A) {row['ans0']}\n"
        f"B) {row['ans1']}\n"
        f"C) {row['ans2']}\n"
        f"Answer:"
    )

def get_label_token_pos(prompt, model, target_word):
    tokens = model.to_tokens(prompt)
    token_strings = [model.to_string(tokens[0][i]) for i in range(tokens.shape[1])]
    for i, tok in enumerate(token_strings):
        if target_word.strip() in tok.strip():
            return i, tokens
    return None, tokens

def get_output_logits(prompt, model):
    tokens = model.to_tokens(prompt)
    with torch.no_grad():
        logits = model(tokens)
    return logits[0, -1, :]

def kl_divergence(logits_a, logits_b):
    p_a = F.softmax(logits_a.float(), dim=-1)
    p_b = F.softmax(logits_b.float(), dim=-1)
    return F.kl_div(p_b.log(), p_a, reduction='sum').item()

def patch_residual_stream(source_prompt, target_prompt,
                           source_label, target_label, layer, model):
    pos_source, source_tokens = get_label_token_pos(source_prompt, model, source_label)
    pos_target, target_tokens = get_label_token_pos(target_prompt, model, target_label)
    if pos_source is None or pos_target is None:
        return None, None, None, False
    with torch.no_grad():
        _, cache = model.run_with_cache(source_tokens)
    source_act = cache[f'blocks.{layer}.hook_resid_post'][0, pos_source, :].clone()
    original_logits = get_output_logits(target_prompt, model)
    def patch_hook(value, hook):
        value[0, pos_target, :] = source_act
        return value
    with torch.no_grad():
        patched_logits = model.run_with_hooks(
            target_tokens,
            fwd_hooks=[(f'blocks.{layer}.hook_resid_post', patch_hook)]
        )[0, -1, :]
    return kl_divergence(original_logits, patched_logits), original_logits, patched_logits, True

print('Helpers defined')

In [ ]:
# Cell 5 — load Pythia-1.4B
print('Loading Pythia-1.4B...')
model = HookedTransformer.from_pretrained(
    'EleutherAI/pythia-1.4b',
    device=DEVICE,
    dtype=torch.float16
)
model.eval()
print('Loaded.')

In [ ]:
# Cell 6 — run patching
print('=' * 60)
print('CAUSAL PATCHING — Pythia-1.4B')
print('=' * 60)

results = []

for layer in TARGET_LAYERS:
    print(f'\nLayer {layer}:')
    kls_h2b, kls_b2h = [], []

    for i in range(N_PAIRS):
        h_prompt = format_prompt(hispanic_rows[i])
        b_prompt = format_prompt(black_rows[i])

        kl, _, _, ok = patch_residual_stream(h_prompt, b_prompt, 'Hispanic', 'Black', layer, model)
        if ok: kls_h2b.append(kl)

        kl, _, _, ok = patch_residual_stream(b_prompt, h_prompt, 'Black', 'Hispanic', layer, model)
        if ok: kls_b2h.append(kl)

    m_h2b, s_h2b = np.mean(kls_h2b), np.std(kls_h2b)
    m_b2h, s_b2h = np.mean(kls_b2h), np.std(kls_b2h)
    print(f'  Hispanic→Black: KL = {m_h2b:.4f} ± {s_h2b:.4f}')
    print(f'  Black→Hispanic: KL = {m_b2h:.4f} ± {s_b2h:.4f}')

    for direction, means, stds, kls in [
        ('Hispanic→Black', m_h2b, s_h2b, kls_h2b),
        ('Black→Hispanic', m_b2h, s_b2h, kls_b2h)
    ]:
        results.append({'model': 'Pythia-1.4B', 'layer': layer,
                        'direction': direction,
                        'mean_kl': round(means, 4),
                        'std_kl': round(stds, 4),
                        'n': len(kls)})

In [ ]:
# Cell 7 — null baseline
print('Null baseline (same-group patch, layer 12):')
baseline_kls = []
for i in range(min(5, len(hispanic_rows) - 1)):
    kl, _, _, ok = patch_residual_stream(
        format_prompt(hispanic_rows[i]),
        format_prompt(hispanic_rows[i+1]),
        'Hispanic', 'Hispanic', 12, model)
    if ok: baseline_kls.append(kl)

baseline_mean = np.mean(baseline_kls)
print(f'  Same-group KL: {baseline_mean:.4f} ± {np.std(baseline_kls):.4f}')

## White-Hispanic causal patching

In [ ]:
# White-Hispanic causal patching — Pythia-1.4B
# Tests whether the White-Hispanic direction shows the same causal profile
# as Hispanic-Black, validating the criminalization-adjacent vocabulary finding

N_PAIRS_WH = 10
PYTHIA_LAYERS = [6, 12, 18, 23]

# Build White rows
white_indices_p = []
for i, row in race_ambig.iterrows():
    ctx = row['context']
    if 'White' in ctx and 'Non-White' not in ctx:
        white_indices_p.append(i)
white_rows_p = race_ambig.iloc[white_indices_p[:N_PAIRS_WH]].reset_index(drop=True)
print(f'White rows: {len(white_rows_p)}')

# Run White-Hispanic patching
print('\nCAUSAL PATCHING — Pythia-1.4B (White-Hispanic direction)')
print('='*60)
wh_results_p = []

for layer in PYTHIA_LAYERS:
    kls_w2h, kls_h2w = [], []
    for i in range(N_PAIRS_WH):
        w_prompt = format_prompt(white_rows_p.iloc[i])
        h_prompt = format_prompt(hispanic_rows[i])

        kl, _, _, ok = patch_residual_stream(w_prompt, h_prompt, 'White', 'Hispanic', layer, model)
        if ok and kl is not None: kls_w2h.append(kl)
        kl, _, _, ok = patch_residual_stream(h_prompt, w_prompt, 'Hispanic', 'White', layer, model)
        if ok and kl is not None: kls_h2w.append(kl)

    for direction, kls in [('White→Hispanic', kls_w2h), ('Hispanic→White', kls_h2w)]:
        mean_kl = np.mean(kls)
        wh_results_p.append({'model': 'Pythia-1.4B', 'layer': layer, 'direction': direction,
                              'mean_kl': round(mean_kl, 4), 'std_kl': round(np.std(kls), 4),
                              'n': len(kls)})
        print(f'  Layer {layer} {direction}: {mean_kl:.4f}')

# Same-group White baseline at layer 6
print('\nSame-group White baseline (layer 6):')
wh_baseline_p = []
for i in range(min(6, len(white_rows_p) - 1)):
    kl, _, _, ok = patch_residual_stream(
        format_prompt(white_rows_p.iloc[i]),
        format_prompt(white_rows_p.iloc[i+1]),
        'White', 'White', 6, model)
    if ok: wh_baseline_p.append(kl)
print(f'  Same-group KL: {np.mean(wh_baseline_p):.4f} \u00b1 {np.std(wh_baseline_p):.4f}')

wh_df_p = pd.DataFrame(wh_results_p)
print('\nFull White-Hispanic table:')
print(wh_df_p.to_string(index=False))

save_path = f'{DRIVE_BASE}/causal_patching/causal_patching_pythia_white_hispanic.csv'
wh_df_p.to_csv(save_path, index=False)
print(f'\nSaved: {save_path}')

# Quick comparison
print('\nCOMPARISON:')
hb_means_p = pd.DataFrame(results).groupby('layer')['mean_kl'].mean()
wh_means_p = wh_df_p.groupby('layer')['mean_kl'].mean()
for layer in PYTHIA_LAYERS:
    print(f'  Layer {layer}: H-B={hb_means_p[layer]:.4f}  W-H={wh_means_p[layer]:.4f}')


In [ ]:
# Cell 8 — summary and save
df = pd.DataFrame(results)
print(df.to_string(index=False))

csv_path = f'{OUTPUT_DIR}/causal_patching_pythia_results.csv'
df.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

# Interpretation
cross_means = [r['mean_kl'] for r in results]
layer_means = {}
for layer in TARGET_LAYERS:
    lr = [r['mean_kl'] for r in results if r['layer'] == layer]
    layer_means[layer] = np.mean(lr)

print('\n' + '=' * 60)
print('INTERPRETATION')
print('=' * 60)
for layer in TARGET_LAYERS:
    print(f'Layer {layer}: mean KL = {layer_means[layer]:.4f} | baseline = {baseline_mean:.4f} | ratio = {layer_means[layer]/max(baseline_mean,0.0001):.1f}x')

In [ ]:
# Cell 9 — figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Causal Patching — Cross-Architecture Comparison\nKL Divergence After Demographic Activation Patch',
             fontsize=12, fontweight='bold')

# GPT-2 reference values from your paper
gpt2_data = {
    'layers': [3, 6, 9, 11],
    'h2b_means': [0.0173, 0.0171, 0.0088, 0.0000],
    'b2h_means': [0.0181, 0.0194, 0.0097, 0.0000],
    'h2b_stds':  [0.0060, 0.0064, 0.0037, 0.0000],
    'b2h_stds':  [0.0096, 0.0102, 0.0048, 0.0000],
    'baseline':   0.0007
}

for ax, (title, data, baseline_val) in zip(axes, [
    ('GPT-2 (117M)', gpt2_data, gpt2_data['baseline']),
    ('Pythia-1.4B (1.4B)', None, baseline_mean)
]):
    if data is None:
        # Pythia — use live results
        h2b_res = [r for r in results if r['direction'] == 'Hispanic→Black']
        b2h_res = [r for r in results if r['direction'] == 'Black→Hispanic']
        layers_plot = [r['layer'] for r in h2b_res]
        h2b_m = [r['mean_kl'] for r in h2b_res]
        b2h_m = [r['mean_kl'] for r in b2h_res]
        h2b_s = [r['std_kl'] for r in h2b_res]
        b2h_s = [r['std_kl'] for r in b2h_res]
    else:
        layers_plot = data['layers']
        h2b_m = data['h2b_means']
        b2h_m = data['b2h_means']
        h2b_s = data['h2b_stds']
        b2h_s = data['b2h_stds']

    x = np.arange(len(layers_plot))
    w = 0.35
    ax.bar(x - w/2, h2b_m, w, yerr=h2b_s, color='#E63946', alpha=0.8,
           capsize=5, label='Hispanic\u2192Black', error_kw={'elinewidth': 1.5})
    ax.bar(x + w/2, b2h_m, w, yerr=b2h_s, color='#457B9D', alpha=0.8,
           capsize=5, label='Black\u2192Hispanic', error_kw={'elinewidth': 1.5})
    ax.axhline(baseline_val, color='gray', linestyle='--', linewidth=1.5,
               label=f'Baseline ({baseline_val:.3f})')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Layer {l}' for l in layers_plot], fontsize=10)
    ax.set_ylabel('KL Divergence', fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    for i, (mh, mb) in enumerate(zip(h2b_m, b2h_m)):
        ax.text(i-w/2, mh + 0.0005, f'{mh:.3f}', ha='center', fontsize=8)
        ax.text(i+w/2, mb + 0.0005, f'{mb:.3f}', ha='center', fontsize=8)

plt.tight_layout()
fig_path = f'{OUTPUT_DIR}/causal_patching_comparison.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()